# Fama-MacBeth Regression for ChronoBERT Stock Return Prediction

This notebook implements the **Fama-MacBeth (1973) cross-sectional regression** framework
from *Chronologically Consistent Large Language Models* (Manela et al., 2025; arXiv:2502.21206).

## Methodology (following the paper)

1. **Monthly cross-sectional Ridge regressions:** Each month $m$, estimate:
$$r_{i,t+1} = \alpha_m + \boldsymbol{\beta}_m' \mathbf{e}_{i,t} + \varepsilon_{i,t+1}$$
where $\mathbf{e}_{i,t}$ is the news embedding for firm $i$ on day $t$.

2. **Expanding-window out-of-sample forecasts:** In month $m'$, average all prior months' coefficients:
$$\hat{r}_{i,t+1} = \bar{\alpha}_{m'} + \bar{\boldsymbol{\beta}}_{m'}' \mathbf{e}_{i,t}$$

3. **Decile portfolio sorts:** Sort stocks into 10 portfolios by predicted return; form a daily-rebalanced, equal-weighted long-short (H-L) portfolio.

4. **Ledoit-Wolf (2008) pairwise Sharpe ratio difference tests:** Formally test whether Sharpe ratio differences between models are statistically significant.

In [ ]:
import os

import pandas as pd
import numpy as np
import yfinance as yf
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import RidgeCV, Ridge
from scipy import stats
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ============================================================
# MODEL REGISTRY
# ============================================================
MODELS = {
    "ChronoBERT_2017": "manelalab/chrono-bert-v1-20171231",
    "ChronoBERT_2018": "manelalab/chrono-bert-v1-20181231",
    "ChronoBERT_2021": "manelalab/chrono-bert-v1-20211231",
    "ChronoBERT_2022": "manelalab/chrono-bert-v1-20221231",
    "Standard_BERT":   "bert-base-uncased",
    "FinBERT":         "yiyanghkust/finbert-pretrain",
}

MODEL_COLORS = {
    "ChronoBERT_2017": "#1b9e77",
    "ChronoBERT_2018": "#66c2a5",
    "ChronoBERT_2021": "#d95f02",
    "ChronoBERT_2022": "#fc8d62",
    "Standard_BERT":   "#7570b3",
    "FinBERT":         "#b2abd2",
}

print(f"Models to evaluate: {list(MODELS.keys())}")

## 0.1 Data Loading & Preprocessing

Load the FNSPID dataset, download price data, compute T+1 forward returns.
Following the paper, we aggregate all news headlines per firm-day before embedding.

In [ ]:
# COVID-affected stock universe
# TOP_TICKERS = [
#     'ZM', 'PTON', 'DOCU', 'NFLX', 'AMZN', 'SHOP', 'TDOC', 'NVDA', 'AMD', 'AAPL',
#     'MSFT', 'GOOGL', 'META', 'ETSY', 'CHWY', 'W', 'ROKU', 'PYPL', 'SQ', 'TEAM',
#     'CRWD', 'OKTA', 'ZS', 'TWLO', 'DDOG', 'FSLY', 'NET', 'SNOW', 'EBAY', 'CRM',
#     'TGT', 'WMT', 'COST', 'HD', 'LOW', 'CLX', 'PG', 'KMB', 'DPZ', 'MRNA',
#     'PFE', 'BNTX', 'JNJ', 'GILD', 'REGN', 'ABT', 'LLY', 'AZN', 'NVAX', 'CCL',
#     'RCL', 'NCLH', 'DAL', 'UAL', 'AAL', 'LUV', 'BA', 'MAR', 'HLT', 'EXPE',
#     'BKNG', 'ABNB', 'UBER', 'LYFT', 'DIS', 'AMC', 'CNK', 'LYV', 'MGM', 'CZR',
#     'WYNN', 'LVS', 'DRI', 'PLAY', 'SYY', 'XOM', 'CVX', 'SLB', 'HAL', 'OXY',
#     'SPG', 'VNO', 'BXP', 'MAC', 'GPS', 'M', 'JWN', 'KSS', 'URBN', 'RTX',
#     'GE', 'JPM', 'BAC', 'WFC', 'C', 'AXP', 'V', 'MA', 'SBUX', 'TSLA'
# ]

# S&P 100 stock universe
TOP_TICKERS = [
    'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'GOOG', 'META', 'BRK-B', 'LLY', 'AVGO',
    'V', 'JPM', 'TSLA', 'WMT', 'UNH', 'MA', 'PG', 'JNJ', 'HD', 'ORCL',
    'MRK', 'COST', 'ABBV', 'BAC', 'CRM', 'CVX', 'NFLX', 'AMD', 'KO', 'PEP',
    'TMO', 'LIN', 'DIS', 'MCD', 'WFC', 'CSCO', 'ABT', 'INTU', 'IBM', 'GE',
    'CAT', 'AXP', 'DHR', 'QCOM', 'TXN', 'VZ', 'AMGN', 'INTC', 'COP', 'NEE',
    'PFE', 'BA', 'SPGI', 'HON', 'UNP', 'RTX', 'LMT', 'LOW', 'GS', 'SYK',
    'BKNG', 'ELV', 'MS', 'PLD', 'MDT', 'C', 'BLK', 'ADP', 'TJX', 'BMY',
    'MMC', 'CB', 'AMT', 'SCHW', 'GILD', 'SBUX', 'MDLZ', 'CI', 'ISRG', 'DE',
    'MO', 'SO', 'DUK', 'CMCSA', 'T', 'REGN', 'ZTS', 'VRTX', 'NOC', 'EOG',
    'BDX', 'SLB', 'BSX', 'EMR', 'CL', 'AON', 'CSX', 'USB', 'F', 'XOM'
]

# Load news
df_news = pd.read_csv("dataset/fnspid_2010_2023.csv", low_memory=False)
df_news['date'] = pd.to_datetime(df_news['Date']).dt.date
df_news = df_news[df_news['Stock_symbol'].isin(TOP_TICKERS)].copy()
df_news.sort_values('date', inplace=True)

# ============================================================
# KEY STEP: Aggregate all headlines per firm-day (following the paper)
# "For each firm-day observation, we aggregate all news articles'
#  headlines for a stock on a given trading day together."
# ============================================================
df_firm_day = (
    df_news
    .groupby(['date', 'Stock_symbol'])['Article_title']
    .apply(lambda titles: ' [SEP] '.join(titles.fillna('').astype(str)))
    .reset_index()
    .rename(columns={'Article_title': 'combined_headlines'})
)

print(f"Raw news articles: {len(df_news)}")
print(f"Firm-day observations: {len(df_firm_day)}")
print(f"Date range: {df_firm_day['date'].min()} to {df_firm_day['date'].max()}")
print(f"Unique tickers: {df_firm_day['Stock_symbol'].nunique()}")

In [ ]:
# Download price data and compute T+1 forward returns
start_date = "2010-01-01"
end_date = "2023-01-01"

prices = yf.download(TOP_TICKERS, start=start_date, end=end_date)['Close']
returns = prices.pct_change(fill_method=None)
forward_returns = returns.shift(-1)

# Reshape to long format
fwd_ret_long = forward_returns.reset_index().melt(
    id_vars='Date', var_name='Stock_symbol', value_name='fwd_return'
)
fwd_ret_long['Date'] = fwd_ret_long['Date'].dt.date
fwd_ret_long.rename(columns={'Date': 'date'}, inplace=True)

# Merge with firm-day aggregated headlines
df_merged = pd.merge(df_firm_day, fwd_ret_long, on=['date', 'Stock_symbol'], how='inner')
df_merged.dropna(subset=['fwd_return'], inplace=True)

# Add year-month for Fama-MacBeth monthly grouping
df_merged['year_month'] = pd.to_datetime(df_merged['date']).dt.to_period('M')

print(f"Final dataset: {len(df_merged)} firm-day observations")
print(f"Date range: {df_merged['date'].min()} to {df_merged['date'].max()}")
print(f"Months: {df_merged['year_month'].nunique()}")
print(f"Avg firms per day: {df_merged.groupby('date')['Stock_symbol'].nunique().mean():.1f}")

## 0.2 Embedding Extraction

Following the paper: *"We process each piece of text through different language models and
extract the hidden states of all tokens. The final embedding for each text is obtained by
averaging the token embeddings."*

We extract embeddings using the **mean of all token hidden states** from the last layer
(the paper tests last-layer, all-layer-average, and first-layer, then selects the best
in real time; we use last-layer mean as a reasonable default).

In [ ]:
def extract_embeddings_mean_pooling(texts, model_name, batch_size=32):
    """
    Extract mean-pooled token embeddings (last hidden layer) from a BERT-style model.
    This follows the paper's approach of averaging token embeddings.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=f"Embedding: {model_name}"):
            batch_texts = texts[i:i+batch_size].tolist()
            inputs = tokenizer(
                batch_texts, padding=True, truncation=True,
                max_length=512, return_tensors="pt"
            ).to(device)
            outputs = model(**inputs)

            # Mean pooling over non-padding tokens
            attention_mask = inputs['attention_mask'].unsqueeze(-1)  # (B, T, 1)
            hidden = outputs.last_hidden_state                       # (B, T, D)
            masked = hidden * attention_mask
            summed = masked.sum(dim=1)                               # (B, D)
            counts = attention_mask.sum(dim=1).clamp(min=1)          # (B, 1)
            mean_emb = (summed / counts).cpu().numpy()
            embeddings.append(mean_emb)

    del model
    torch.cuda.empty_cache()
    return np.vstack(embeddings)

# Extract embeddings for all models on the aggregated firm-day headlines
texts = df_merged['combined_headlines'].values
for model_name, model_path in MODELS.items():
    print(f"\n--- Extracting: {model_name} ---")
    embs = extract_embeddings_mean_pooling(texts, model_path)
    df_merged[model_name] = list(embs)
    print(f"  Shape: {embs.shape}")

print(f"\nAll embeddings extracted. DataFrame: {df_merged.shape}")

---
## 1. Fama-MacBeth Cross-Sectional Ridge Regression

The key difference from a simple rolling-window regression:

- **Monthly estimation:** Each calendar month $m$, we pool ALL firm-day observations within
  that month and estimate a single cross-sectional Ridge regression.
- **Expanding-window coefficient averaging:** To predict in month $m'$, we average the
  intercepts and slope vectors from ALL prior months $1, \ldots, m'-1$. This is the
  Fama-MacBeth "second pass" — it creates stable, out-of-sample forecasts.
- **Leave-one-out CV for Ridge penalty:** Following the paper, we use LOOCV to select
  $\lambda$ from a grid of $10^{-10}$ to $10^{10}$.

In [ ]:
# ============================================================
# FAMA-MACBETH RIDGE REGRESSION ENGINE
# ============================================================

# Ridge penalty grid (following the paper: 10^{-10} to 10^{10})
ALPHA_GRID = np.logspace(-10, 10, num=21)

def fama_macbeth_ridge(df, embedding_col, burn_in_months=1):
    """
    Fama-MacBeth cross-sectional Ridge regression with expanding-window averaging.
    
    Parameters
    ----------
    df : DataFrame with columns: date, year_month, Stock_symbol, fwd_return, embedding_col
    embedding_col : column name containing embedding vectors
    burn_in_months : number of months used to fit before we start predicting OOS
    
    Returns
    -------
    daily_predictions : DataFrame with columns [date, Stock_symbol, predicted_return, actual_return]
    monthly_coefficients : list of (alpha_m, beta_m) tuples for each month
    monthly_lambdas : list of selected Ridge lambda per month
    """
    months = sorted(df['year_month'].unique())
    
    # Step 1: Estimate monthly cross-sectional Ridge regressions
    monthly_alphas = []    # intercepts
    monthly_betas = []     # coefficient vectors
    monthly_lambdas = []   # selected Ridge penalties
    monthly_labels = []    # month labels
    
    print(f"Step 1: Estimating monthly cross-sectional regressions ({len(months)} months)...")
    
    for month in tqdm(months, desc="Monthly Ridge"):
        month_data = df[df['year_month'] == month]
        
        if len(month_data) < 20:
            continue
        
        X = np.vstack(month_data[embedding_col].values)
        y = month_data['fwd_return'].values
        
        # RidgeCV with leave-one-out cross-validation
        ridge_cv = RidgeCV(
            alphas=ALPHA_GRID,
            cv=None,  # None = efficient LOOCV for Ridge
            scoring='neg_mean_squared_error'
        )
        ridge_cv.fit(X, y)
        
        monthly_alphas.append(ridge_cv.intercept_)
        monthly_betas.append(ridge_cv.coef_)
        monthly_lambdas.append(ridge_cv.alpha_)
        monthly_labels.append(month)
    
    print(f"  Estimated {len(monthly_alphas)} monthly regressions")
    print(f"  Ridge lambda range: [{min(monthly_lambdas):.2e}, {max(monthly_lambdas):.2e}]")
    
    # Step 2: Generate expanding-window OOS predictions
    print(f"\nStep 2: Generating out-of-sample predictions (burn-in = {burn_in_months} months)...")
    
    all_predictions = []
    
    for m_idx in range(burn_in_months, len(monthly_labels)):
        current_month = monthly_labels[m_idx]
        
        # Average ALL prior months' coefficients (expanding window)
        avg_alpha = np.mean(monthly_alphas[:m_idx])
        avg_beta = np.mean(monthly_betas[:m_idx], axis=0)
        
        # Get current month's data for prediction
        month_data = df[df['year_month'] == current_month].copy()
        
        if len(month_data) == 0:
            continue
        
        X_test = np.vstack(month_data[embedding_col].values)
        predictions = avg_alpha + X_test @ avg_beta
        
        month_data = month_data[['date', 'Stock_symbol', 'fwd_return']].copy()
        month_data['predicted_return'] = predictions
        all_predictions.append(month_data)
    
    daily_preds = pd.concat(all_predictions, ignore_index=True)
    print(f"  Total OOS predictions: {len(daily_preds)}")
    print(f"  Date range: {daily_preds['date'].min()} to {daily_preds['date'].max()}")
    
    return daily_preds, list(zip(monthly_alphas, monthly_betas)), monthly_lambdas

print("Fama-MacBeth engine defined.")

In [ ]:
# ============================================================
# Run Fama-MacBeth for all models
# ============================================================
fm_results = {}  # model_name -> daily_predictions DataFrame

for model_name in MODELS.keys():
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    preds, coeffs, lambdas = fama_macbeth_ridge(df_merged, model_name)
    fm_results[model_name] = preds

print("\n\nAll Fama-MacBeth regressions complete.")

---
## 2. Decile Portfolio Sorts (Table 7 Replication)

Following the paper:
- Each trading day, sort all stocks with predictions into **10 equal-sized portfolios** based
  on the out-of-sample predicted return.
- Compute equal-weighted actual returns for each decile.
- The **H-L (High minus Low)** portfolio goes long decile 10 and short decile 1.
- Report annualised mean return, standard deviation, and Sharpe ratio.

In [ ]:
# ============================================================
# DECILE PORTFOLIO CONSTRUCTION
# ============================================================

def build_decile_portfolios(daily_preds, n_deciles=10):
    """
    Sort stocks into decile portfolios each day and compute returns.
    
    Parameters
    ----------
    daily_preds : DataFrame with [date, Stock_symbol, predicted_return, fwd_return]
    n_deciles : number of portfolio buckets (default: 10 for deciles)
    
    Returns
    -------
    decile_returns : DataFrame with daily returns for each decile + H-L
    decile_summary : DataFrame with annualised mean, std, Sharpe for each decile
    """
    daily_decile_returns = []
    
    for date, group in daily_preds.groupby('date'):
        if len(group) < n_deciles:
            continue
        
        # Assign decile labels (1=lowest predicted, 10=highest predicted)
        group = group.copy()
        group['decile'] = pd.qcut(
            group['predicted_return'],
            q=n_deciles,
            labels=False,
            duplicates='drop'
        ) + 1  # shift from 0-based to 1-based
        
        # Equal-weighted return per decile
        decile_ret = group.groupby('decile')['fwd_return'].mean()
        decile_ret.name = date
        daily_decile_returns.append(decile_ret)
    
    # Stack into a DataFrame: rows=dates, columns=deciles
    decile_df = pd.DataFrame(daily_decile_returns)
    decile_df.index = pd.to_datetime(decile_df.index)
    decile_df.columns = [int(c) for c in decile_df.columns]
    
    # Add H-L (High minus Low) column
    max_decile = decile_df.columns.max()
    min_decile = decile_df.columns.min()
    decile_df['H-L'] = decile_df[max_decile] - decile_df[min_decile]
    
    # Summary statistics (annualised)
    summary_rows = []
    for col in decile_df.columns:
        ret = decile_df[col].dropna()
        ann_mean = ret.mean() * 252 * 100       # in percentage points
        ann_std = ret.std() * np.sqrt(252) * 100  # in percentage points
        sharpe = (ret.mean() / ret.std()) * np.sqrt(252) if ret.std() > 0 else 0
        label = f"D{col}" if isinstance(col, int) else col
        summary_rows.append({
            'Portfolio': label,
            'Mean (%)': ann_mean,
            'SD (%)': ann_std,
            'Sharpe': sharpe
        })
    
    summary_df = pd.DataFrame(summary_rows).set_index('Portfolio')
    return decile_df, summary_df

# Build decile portfolios for all models
portfolio_results = {}  # model_name -> (decile_df, summary_df)

for model_name, preds in fm_results.items():
    decile_df, summary_df = build_decile_portfolios(preds)
    portfolio_results[model_name] = (decile_df, summary_df)

print("Decile portfolios constructed for all models.")

In [ ]:
# ============================================================
# TABLE 7 REPLICATION: Full Decile Performance Table
# ============================================================
for model_name in MODELS.keys():
    _, summary = portfolio_results[model_name]
    print(f"\n{'='*60}")
    print(f"  {model_name} — Decile Portfolio Performance")
    print(f"{'='*60}")
    print(summary.to_string(float_format=lambda x: f"{x:8.2f}"))
    print()

In [ ]:
# ============================================================
# COMPACT SUMMARY: H-L Portfolio Comparison (Table 7 bottom row)
# ============================================================
hl_summary_rows = []
for model_name in MODELS.keys():
    _, summary = portfolio_results[model_name]
    hl = summary.loc['H-L']
    hl_summary_rows.append({
        'Model': model_name,
        'H-L Mean (%)': hl['Mean (%)'],
        'H-L SD (%)': hl['SD (%)'],
        'H-L Sharpe': hl['Sharpe']
    })

hl_summary = pd.DataFrame(hl_summary_rows).set_index('Model')
hl_summary = hl_summary.sort_values('H-L Sharpe', ascending=False)

print("=" * 60)
print("  H-L (Long-Short) Portfolio Summary")
print("=" * 60)
print(hl_summary.to_string(float_format=lambda x: f"{x:.3f}"))
print()

# Also compute Fama-MacBeth t-statistics for the H-L mean returns
print("\nFama-MacBeth t-statistics (H0: mean H-L return = 0):")
print("-" * 50)
for model_name in MODELS.keys():
    decile_df, _ = portfolio_results[model_name]
    hl_returns = decile_df['H-L'].dropna()
    t_stat, p_val = stats.ttest_1samp(hl_returns, 0)
    print(f"  {model_name:20s}  t = {t_stat:7.3f}  p = {p_val:.4f}  {'***' if p_val < 0.01 else '**' if p_val < 0.05 else '*' if p_val < 0.10 else ''}")

---
## 3. Ledoit-Wolf (2008) Pairwise Sharpe Ratio Tests (Table 8 Replication)

A key contribution of the paper is formally testing whether Sharpe ratio differences are
statistically significant using the **Ledoit & Wolf (2008)** HAC-robust test.

The test uses a HAC (Heteroskedasticity and Autocorrelation Consistent) estimator for the
variance of the Sharpe ratio difference, which is critical for financial time series that
exhibit serial correlation and heteroskedasticity.

**Null hypothesis:** $H_0: SR_{\text{row}} \leq SR_{\text{col}}$
(the row model's Sharpe is no larger than the column model's Sharpe)

Low p-values in the upper triangle indicate the row model significantly outperforms the column model.

In [ ]:
# ============================================================
# LEDOIT-WOLF (2008) SHARPE RATIO DIFFERENCE TEST
# ============================================================
# 
# Reference: Ledoit, O. & Wolf, M. (2008). "Robust performance hypothesis
# testing with the Sharpe ratio." Journal of Empirical Finance, 15(5), 850-859.
#
# The test statistic uses a HAC variance estimator for the difference
# Delta = SR_1 - SR_2, computed from the joint distribution of the
# two return series.

def hac_variance(x, max_lag=None):
    """
    Newey-West HAC variance estimator for a time series x.
    Uses the Bartlett kernel with automatic bandwidth selection.
    """
    T = len(x)
    if max_lag is None:
        max_lag = int(np.floor(T ** (1/3)))  # Andrews (1991) rule of thumb
    
    x_demean = x - x.mean()
    
    # Gamma_0
    gamma_0 = np.dot(x_demean, x_demean) / T
    
    # Sum of autocovariances with Bartlett weights
    weighted_sum = 0.0
    for j in range(1, max_lag + 1):
        gamma_j = np.dot(x_demean[j:], x_demean[:-j]) / T
        weight = 1 - j / (max_lag + 1)  # Bartlett kernel
        weighted_sum += 2 * weight * gamma_j
    
    return gamma_0 + weighted_sum


def ledoit_wolf_test(ret_a, ret_b):
    """
    Ledoit-Wolf (2008) test for H0: SR_a <= SR_b.
    
    Parameters
    ----------
    ret_a, ret_b : array-like, daily return series (must be same length, aligned)
    
    Returns
    -------
    p_value : one-sided p-value for H0: SR_a <= SR_b
    z_stat  : test statistic (positive = a outperforms b)
    """
    ret_a = np.asarray(ret_a)
    ret_b = np.asarray(ret_b)
    T = len(ret_a)
    
    mu_a, mu_b = ret_a.mean(), ret_b.mean()
    sig_a, sig_b = ret_a.std(ddof=0), ret_b.std(ddof=0)
    sr_a = mu_a / sig_a if sig_a > 0 else 0
    sr_b = mu_b / sig_b if sig_b > 0 else 0
    
    # Construct the "influence function" for the Sharpe difference
    # Following Ledoit-Wolf (2008) Equation (4):
    # Let theta = (mu_a, mu_b, mu_a2, mu_b2, mu_ab)
    # The Sharpe ratio difference is Delta(theta)
    
    # Compute raw moments
    mu_a2 = (ret_a ** 2).mean()
    mu_b2 = (ret_b ** 2).mean()
    mu_ab = (ret_a * ret_b).mean()
    
    # Gradient of the Sharpe difference w.r.t. theta = (mu_a, mu_b, mu_a2, mu_b2, mu_ab)
    var_a = mu_a2 - mu_a ** 2
    var_b = mu_b2 - mu_b ** 2
    
    if var_a <= 0 or var_b <= 0:
        return 0.5, 0.0
    
    # Partial derivatives (see Ledoit-Wolf 2008, proof of Proposition 1)
    d1 = 1.0 / np.sqrt(var_a)                              # d(Delta)/d(mu_a)
    d2 = -1.0 / np.sqrt(var_b)                             # d(Delta)/d(mu_b)
    d3 = -0.5 * mu_a / (var_a ** 1.5)                      # d(Delta)/d(mu_a2)
    d4 = 0.5 * mu_b / (var_b ** 1.5)                       # d(Delta)/d(mu_b2)
    d5 = 0.0                                                 # d(Delta)/d(mu_ab) = 0
    
    grad = np.array([d1, d2, d3, d4, d5])
    
    # Build the T x 5 matrix of "scores"
    scores = np.column_stack([
        ret_a - mu_a,
        ret_b - mu_b,
        ret_a ** 2 - mu_a2,
        ret_b ** 2 - mu_b2,
        ret_a * ret_b - mu_ab
    ])
    
    # Influence function for Delta
    psi = scores @ grad  # (T,)
    
    # HAC variance of psi
    v_hat = hac_variance(psi)
    
    # Standard error
    se = np.sqrt(v_hat / T)
    
    if se <= 0:
        return 0.5, 0.0
    
    # Test statistic
    delta_hat = sr_a - sr_b
    z_stat = delta_hat / se
    
    # One-sided p-value: P(Z > z) under H0
    p_value = 1 - stats.norm.cdf(z_stat)
    
    return p_value, z_stat

print("Ledoit-Wolf (2008) test function defined.")

In [ ]:
# ============================================================
# TABLE 8 REPLICATION: Pairwise Sharpe Ratio Difference Tests
# ============================================================
# Entry (i,j) = p-value for H0: SR_i <= SR_j
# Low p-value in upper triangle = row model significantly outperforms column model

model_names = list(MODELS.keys())
n = len(model_names)

# First align all H-L return series on common dates
hl_returns = {}
for mname in model_names:
    decile_df, _ = portfolio_results[mname]
    hl_returns[mname] = decile_df['H-L'].dropna()

# Find common dates across all models
common_idx = hl_returns[model_names[0]].index
for mname in model_names[1:]:
    common_idx = common_idx.intersection(hl_returns[mname].index)

print(f"Common trading days across all models: {len(common_idx)}")

# Build pairwise p-value matrix
pval_matrix = np.full((n, n), np.nan)
zstat_matrix = np.full((n, n), np.nan)

for i in range(n):
    for j in range(n):
        if i == j:
            continue
        ret_i = hl_returns[model_names[i]].loc[common_idx].values
        ret_j = hl_returns[model_names[j]].loc[common_idx].values
        p_val, z_stat = ledoit_wolf_test(ret_i, ret_j)
        pval_matrix[i, j] = p_val
        zstat_matrix[i, j] = z_stat

pval_df = pd.DataFrame(pval_matrix, index=model_names, columns=model_names)

print("\n" + "=" * 70)
print("  Ledoit-Wolf (2008) Pairwise Sharpe Ratio Difference Test")
print("  H0: SR(row) <= SR(col)  |  Low p-value = row outperforms column")
print("=" * 70)
print(pval_df.to_string(float_format=lambda x: f"{x:.3f}" if not np.isnan(x) else "  --", na_rep="  --"))

In [ ]:
# ============================================================
# TABLE 8 HEATMAP VISUALISATION
# ============================================================
fig, ax = plt.subplots(figsize=(10, 8))

# Mask diagonal
mask = np.eye(n, dtype=bool)
pval_plot = pval_df.copy()

sns.heatmap(pval_plot, annot=True, fmt='.3f', cmap='RdYlGn_r',
            vmin=0, vmax=1, mask=mask, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'p-value'},
            annot_kws={'fontsize': 10})

ax.set_title('Ledoit-Wolf (2008) Pairwise Sharpe Ratio Test\n'
             r'$H_0$: SR(row) $\leq$ SR(col)  |  Green = row outperforms',
             fontsize=12)
plt.tight_layout()
plt.savefig('figures/fm_pairwise_sharpe_test.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Performance Visualisations

In [ ]:
# ============================================================
# 4a: Cumulative Returns of H-L Portfolios + Sharpe CI
# ============================================================

def bootstrap_sharpe(returns, n_boot=5000, seed=42):
    """Return (sharpe, ci_low, ci_high) using bootstrap."""
    rng = np.random.default_rng(seed)
    sr = np.sqrt(252) * returns.mean() / returns.std()
    arr = returns.values
    boot_srs = []
    for _ in range(n_boot):
        sample = rng.choice(arr, size=len(arr), replace=True)
        s = np.std(sample, ddof=1)
        boot_srs.append(np.sqrt(252) * sample.mean() / s if s > 0 else 0.0)
    ci_lo, ci_hi = np.percentile(boot_srs, [2.5, 97.5])
    return sr, ci_lo, ci_hi

# --- compute Sharpe + CI for every model ---
sharpe_data = {}
for model_name in MODELS.keys():
    decile_df, _ = portfolio_results[model_name]
    hl = decile_df['H-L'].dropna()
    sr, lo, hi = bootstrap_sharpe(hl)
    sharpe_data[model_name] = {'sharpe': sr, 'ci_lo': lo, 'ci_hi': hi}

# sort by Sharpe ascending (bottom = lowest, top = highest)
sorted_models = sorted(sharpe_data, key=lambda m: sharpe_data[m]['sharpe'])

# --- figure ---
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(18, 7))

# Left: Cumulative Returns
for model_name in MODELS.keys():
    decile_df, _ = portfolio_results[model_name]
    hl = decile_df['H-L'].dropna()
    cum_ret = (1 + hl).cumprod() - 1
    ax_left.plot(cum_ret.index, cum_ret, label=model_name,
                 color=MODEL_COLORS[model_name], linewidth=1.5)

ax_left.axvspan(pd.to_datetime('2020-02-19'), pd.to_datetime('2020-03-23'),
                color='red', alpha=0.1, label='COVID-19 Crash')
ax_left.set_title('(a) Cumulative Returns of Long-Short Portfolio', fontsize=12)
ax_left.set_ylabel('Cumulative Return')
ax_left.set_xlabel('Date')
ax_left.legend(loc='upper left', fontsize=9)
ax_left.grid(True, alpha=0.3)
ax_left.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()

# Right: Sharpe Ratio with 95% Bootstrap CI
y_pos = list(range(len(sorted_models)))
for i, model_name in enumerate(sorted_models):
    d = sharpe_data[model_name]
    sr, lo, hi = d['sharpe'], d['ci_lo'], d['ci_hi']
    color = MODEL_COLORS[model_name]
    ax_right.barh(i, sr, color=color, alpha=0.85, edgecolor='none')
    ax_right.errorbar(sr, i,
                      xerr=[[sr - lo], [hi - sr]],
                      fmt='none', color='black', capsize=4, linewidth=1.2)

ax_right.axvline(x=0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
ax_right.set_yticks(y_pos)
ax_right.set_yticklabels(sorted_models, fontsize=10)
ax_right.set_xlabel('Annualised Sharpe Ratio', fontsize=11)
ax_right.set_title('(b) Sharpe Ratio with 95% Bootstrap CI', fontsize=12)
ax_right.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/fm_cumulative_HL.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 4a (cont): Bootstrap Sharpe Ratio Table (10,000 resamples)
# ============================================================

def bootstrap_sharpe_table(returns, n_boot=10_000, seed=42):
    """Return (sharpe, ci_low, ci_high) via IID bootstrap."""
    rng = np.random.default_rng(seed)
    arr = returns.values
    sr = np.sqrt(252) * arr.mean() / arr.std(ddof=1)
    boot = np.array([
        rng.choice(arr, size=len(arr), replace=True)
        for _ in range(n_boot)
    ])
    boot_sr = np.sqrt(252) * boot.mean(axis=1) / boot.std(axis=1, ddof=1)
    ci_lo, ci_hi = np.percentile(boot_sr, [2.5, 97.5])
    return sr, ci_lo, ci_hi

print('Bootstrap Sharpe Ratios (95% CI, 10,000 resamples)')
print('-' * 65)
for model_name in MODELS.keys():
    decile_df, _ = portfolio_results[model_name]
    hl = decile_df['H-L'].dropna()
    sr, lo, hi = bootstrap_sharpe_table(hl)
    print(f'  {model_name:<22}  Sharpe = {sr:+.3f}  [{lo:+.3f}, {hi:+.3f}]')


In [ ]:
# ============================================================
# 4b: Monotonicity Plot — Mean Return by Decile (all models)
# ============================================================
# The paper's Table 7 shows a clear monotonic pattern from D1 to D10.
# This plot visualises that pattern for each model.

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)
axes = axes.flatten()

for idx, model_name in enumerate(MODELS.keys()):
    ax = axes[idx]
    _, summary = portfolio_results[model_name]
    
    # Extract decile means (exclude H-L)
    decile_means = summary.loc[[f'D{d}' for d in range(1, 11)], 'Mean (%)']
    decile_labels = [f'D{d}' for d in range(1, 11)]
    
    colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, 10))
    ax.bar(decile_labels, decile_means.values, color=colors, edgecolor='black', linewidth=0.5)
    ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
    
    hl_sharpe = summary.loc['H-L', 'Sharpe']
    ax.set_title(f'{model_name}\nH-L Sharpe = {hl_sharpe:.2f}', fontsize=11)
    ax.set_ylabel('Annualised Mean Return (%)')
    ax.grid(True, axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Decile Portfolio Mean Returns (Fama-MacBeth Ridge)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fm_decile_monotonicity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 4c: Sharpe Ratio Bar Chart (H-L only)
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))

sharpes = hl_summary['H-L Sharpe']
colors = [MODEL_COLORS[m] for m in sharpes.index]

bars = ax.bar(sharpes.index, sharpes.values, color=colors, edgecolor='black', linewidth=0.5)

# Add value labels on bars
for bar, val in zip(bars, sharpes.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Annualised Sharpe Ratio', fontsize=11)
ax.set_title('H-L Portfolio Sharpe Ratios (Fama-MacBeth Ridge)', fontsize=13)
ax.grid(True, axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('figures/fm_sharpe_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Sub-Period Analysis & Chronological Consistency Tests

Following the paper's Figure 3 analysis: *"The real-time model's Sharpe ratio consistently
outperforms most vintage models"* — we test whether look-ahead bias manifests differently
across market regimes, and whether the "envelope" phenomenon holds.

In [ ]:
# ============================================================
# 5a: Sub-Period Sharpe Ratios
# ============================================================
SUB_PERIODS = {
    'Pre-COVID':   (pd.to_datetime('2019-01-01'), pd.to_datetime('2020-02-18')),
    'COVID Crash':  (pd.to_datetime('2020-02-19'), pd.to_datetime('2020-06-30')),
    'Recovery':     (pd.to_datetime('2020-07-01'), pd.to_datetime('2021-06-30')),
    'Post-COVID':   (pd.to_datetime('2021-07-01'), pd.to_datetime('2022-12-31')),
}

period_results = []

for model_name in MODELS.keys():
    decile_df, _ = portfolio_results[model_name]
    hl = decile_df['H-L'].dropna()
    
    for period_name, (start, end) in SUB_PERIODS.items():
        mask = (hl.index >= start) & (hl.index <= end)
        period_ret = hl[mask]
        if len(period_ret) < 10:
            continue
        
        mu = period_ret.mean()
        sigma = period_ret.std()
        sharpe = np.sqrt(252) * (mu / sigma) if sigma > 0 else 0
        
        period_results.append({
            'Model': model_name,
            'Period': period_name,
            'Sharpe': sharpe,
            'Ann. Return (%)': mu * 252 * 100,
            'N Days': len(period_ret)
        })

period_df = pd.DataFrame(period_results)
pivot = period_df.pivot(index='Model', columns='Period', values='Sharpe')
pivot = pivot[['Pre-COVID', 'COVID Crash', 'Recovery', 'Post-COVID']]

print("H-L Sharpe Ratios by Sub-Period (Fama-MacBeth)")
print("=" * 70)
print(pivot.to_string(float_format=lambda x: f"{x:+.3f}"))

In [ ]:
# ============================================================
# 5b: Sub-Period Grouped Bar Chart
# ============================================================
fig, ax = plt.subplots(figsize=(14, 6))

periods = ['Pre-COVID', 'COVID Crash', 'Recovery', 'Post-COVID']
x = np.arange(len(periods))
width = 0.12
n_models = len(MODELS)

for i, model_name in enumerate(MODELS.keys()):
    vals = []
    for p in periods:
        val = pivot.loc[model_name, p] if model_name in pivot.index else 0
        vals.append(val)
    offset = (i - n_models / 2 + 0.5) * width
    ax.bar(x + offset, vals, width, label=model_name,
           color=MODEL_COLORS[model_name], edgecolor='black', linewidth=0.5)

ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(periods, fontsize=11)
ax.set_ylabel('Annualised Sharpe Ratio (H-L)', fontsize=11)
ax.set_title('H-L Sharpe by Market Regime (Fama-MacBeth)', fontsize=13)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/fm_subperiod_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5c: Rolling Sharpe (Figure 3 analogue from the paper)
# ============================================================
# The paper's Figure 3 plots cumulative Sharpe or performance per vintage.
# We plot rolling 63-day Sharpe to show the "envelope" phenomenon.

fig, ax = plt.subplots(figsize=(15, 6))
window = 63

for model_name in MODELS.keys():
    decile_df, _ = portfolio_results[model_name]
    hl = decile_df['H-L'].dropna()
    rolling_mu = hl.rolling(window).mean()
    rolling_std = hl.rolling(window).std()
    rolling_sharpe = np.sqrt(252) * rolling_mu / rolling_std
    ax.plot(rolling_sharpe.index, rolling_sharpe, label=model_name,
            color=MODEL_COLORS[model_name], linewidth=1.2, alpha=0.85)

# Regime shading
for pname, (start, end) in SUB_PERIODS.items():
    if pname == 'COVID Crash':
        ax.axvspan(start, end, color='red', alpha=0.08)
    elif pname == 'Recovery':
        ax.axvspan(start, end, color='green', alpha=0.05)

ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Rolling 63-Day Sharpe Ratio — H-L Portfolio (Fama-MacBeth)', fontsize=13)
ax.set_ylabel('Annualised Sharpe (63d rolling)')
ax.set_xlabel('Date')
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()

plt.tight_layout()
plt.savefig('figures/fm_rolling_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5d: Sub-Period Ledoit-Wolf Tests (COVID Crash focus)
# ============================================================
# Test whether Sharpe differences are significant within specific sub-periods

print("Ledoit-Wolf Tests Within Sub-Periods")
print("=" * 70)

for period_name, (start, end) in SUB_PERIODS.items():
    print(f"\n--- {period_name} ---")
    
    # Get aligned H-L returns for this sub-period
    period_hl = {}
    for mname in MODELS.keys():
        decile_df, _ = portfolio_results[mname]
        hl = decile_df['H-L'].dropna()
        mask = (hl.index >= start) & (hl.index <= end)
        period_hl[mname] = hl[mask]
    
    # Find common dates
    common = period_hl[list(MODELS.keys())[0]].index
    for mname in list(MODELS.keys())[1:]:
        common = common.intersection(period_hl[mname].index)
    
    if len(common) < 20:
        print(f"  Insufficient data ({len(common)} days)")
        continue
    
    # Test ChronoBERT_2018 vs ChronoBERT_2022 (key comparison)
    for pair in [('ChronoBERT_2018', 'ChronoBERT_2022'),
                 ('ChronoBERT_2018', 'Standard_BERT'),
                 ('ChronoBERT_2022', 'FinBERT')]:
        m1, m2 = pair
        r1 = period_hl[m1].loc[common].values
        r2 = period_hl[m2].loc[common].values
        p_val, z_stat = ledoit_wolf_test(r1, r2)
        sr1 = np.sqrt(252) * r1.mean() / r1.std() if r1.std() > 0 else 0
        sr2 = np.sqrt(252) * r2.mean() / r2.std() if r2.std() > 0 else 0
        sig = '***' if p_val < 0.01 else '**' if p_val < 0.05 else '*' if p_val < 0.10 else ''
        print(f"  {m1} vs {m2}: SR={sr1:.2f} vs {sr2:.2f}, p={p_val:.3f} {sig}")

---
## 6. Discussion

### Methodology Differences

| Aspect | Original Notebook | This Notebook (Paper's Method) |
|--------|------------------|-------------------------------|
| **Regression** | Rolling 60-day window | Monthly Fama-MacBeth cross-sectional |
| **Coefficient estimation** | Per-window fit, discard | Expanding-window average of monthly coefficients |
| **Ridge penalty selection** | Fixed $\alpha = 1.0$ | Leave-one-out CV per month ($10^{-10}$ to $10^{10}$) |
| **News aggregation** | Individual headlines (one row per article) | Firm-day aggregation (all headlines concatenated per stock-day) |
| **Embedding** | [CLS] token only | Mean pooling over all tokens (following paper) |
| **Portfolio** | Top/bottom 10% (quintile-like) | Proper decile sorts (10 portfolios) |
| **Statistical test** | None / ad-hoc Sharpe comparison | Ledoit-Wolf (2008) HAC-robust pairwise test |
| **Evaluation** | Sharpe ratio only | Full decile table (Mean, SD, SR) + t-statistics |

### Why Fama-MacBeth Matters

The Fama-MacBeth (1973) framework is the gold standard for cross-sectional asset pricing tests because:

1. **Proper cross-sectional structure:** Each month's regression captures how embeddings relate
   to returns *across firms* — not across time. This avoids temporal autocorrelation in the
   training signal.

2. **Expanding-window averaging:** Instead of discarding old estimates, we average over all prior
   months. This stabilises the coefficient estimates and produces smoother out-of-sample predictions,
   analogous to the Bayesian shrinkage effect of the paper's approach.

3. **Standard errors:** The time-series of monthly coefficient estimates directly yields
   Fama-MacBeth t-statistics for whether the embedding dimensions carry a cross-sectional
   return premium. This is the correct frequentist test for "do these features predict returns?"

### Why Ledoit-Wolf (2008) Matters

Standard Sharpe ratio comparisons assume i.i.d. returns. Financial returns exhibit:
- **Fat tails** (kurtosis >> 3)
- **Volatility clustering** (ARCH effects)
- **Serial correlation** in long-short portfolio returns

The Ledoit-Wolf test uses a HAC (Heteroskedasticity and Autocorrelation Consistent) variance
estimator — specifically, a Newey-West estimator with Bartlett kernel — to produce p-values
that account for these violations. Without this correction, standard tests dramatically
understate uncertainty.

### Comparison to the Paper's Results

The paper (Table 7) reports H-L Sharpe ratios around **4.0-4.9** using the Dow Jones Newswire
dataset (2008-2023). Our results will differ because we use:
- **FNSPID** (free) instead of Dow Jones Newswire (proprietary)
- **100 COVID-affected stocks** instead of the full CRSP universe
- **2019-2022** instead of 2008-2023

These differences are expected and documented. The key comparison is the *relative ordering*
of models and whether the Ledoit-Wolf tests show the same pattern of statistical significance.